# Assignment 1: Rosa's Pizza


# Getting Started

In [17]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [18]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

# Use a fixed seed so the numbers are the same every time the notebook runs
SEED = 1

print("Zones:", ZONES)
print("Time blocks:", TIME_BLOCKS)
print("Costs:", COSTS)
print("Current promise:", PROMISE, "minutes")

Zones: ['Central', 'North', 'Far West']
Time blocks: ['Lunch', 'Weekday eve', 'Fri/Sat eve', 'Other']
Costs: {'refund': 10.0, 'churn_orders': 1.8, 'margin': 9.0}
Current promise: 45 minutes


In [19]:
# Quick check that the starter package works
sample = delivery_times("Far West", "Fri/Sat eve", 45)
print(sample)
print(len(sample), "orders; average delivery time", round(sample.mean(), 1), "minutes")

[48.2 49.2 48.5 45.7 43.6 34.5 41.9 19.8 80.1 66.9 56.  38.3 48.4 49.9
 22.5 29.3 49.6 51.6 36.3 27.5 65.4 65.5 47.7 56.6 37.7 54.  33.3 67.5
 33.1 36.3 27.4 36.9 56.7 55.9 33.9 50.4 36.6 37.3 37.5 33.5 27.1 37.8
 45.7 33.2 52.6 54.2 36.  52.3 26.3 42.4 28.  18.  76.7 75.3 33.8 59.3
 56.1 42.  35.7 68.9 63.1 71.5 37.  81.8 23.1 55.7 26.6 31.3 47.4 53.9
 38.8 75.9 31.3 21.5 41.5 58.  51.  36.7 62.5 54.6 44.2 67.6 42.5 41.8
 34.3 37.5 35.4 35.5 24.6 35.8 19.4 22.6 25.5 19.2 61.7 56.2 51.9 74.4
 43.8 64.4 56.8 39.  31.3 28.6 50.  44.2 42.3 27.7 51.2 57.2 37.9 45.8
 44.6 65.2 40.9 44.  53.3 27.7 32.3 52.5 31.4 53.3 57.1 29.2 34.4 51.3
 33.6 35.5 30.4 41.9 58.6 50.6 40.1 42.2 38.1 25.6 29.  50.4 64.5 61.1
 44.5 65.2 35.5 34.3 54.7 35.5 52.7 50.2 28.3 34.4 23.2 40.1 35.3 73.7
 41.3 53.7 43.8 32.2 40.3 29.2 51.4 34.3 22.8 50.7 43.9 51.3 29.3 44.5
 38.5 37.4 49.  39.5 71.2 50.4 64.4 27.5 33.8 50.7 44.2 50.4 27.7 49.1
 59.7 22.2 66.4 45.9 82.3 78.7 35.3 34.1 47.1 42.  32.5 40.2 38.  43.7
 61.3 

# Part I. Describe: How bad is it, really?

## Part 1a: Percentage of late orders

First I wrote two small helper functions. If the user types `"all"`, they return the full list of zones (or time blocks). Otherwise they return a list with just the one name. This means the same loop works for all four cases.

The main function then goes through every zone and time block it needs, adds up the total number of orders and counts how many took longer than the promise.

In [20]:
def pick_zones(zone):
    # "all" means every zone; otherwise wrap the single zone in a list
    # so the caller can loop over the result either way
    if zone == "all":
        return ZONES
    else:
        return [zone]


def pick_blocks(block):
    # same idea as pick_zones, but for time blocks
    if block == "all":
        return TIME_BLOCKS
    else:
        return [block]


def late_percentage(zone, block, promise_min):
    # works out what % of orders arrive later than the promised time
    # zone and block can each be one value or "all"
    total_orders = 0
    late_orders = 0

    # go through every zone and time block combination we picked
    for z in pick_zones(zone):
        for b in pick_blocks(block):
            # simulate delivery times for this zone and block
            # (a fixed SEED gives the same results on every run)
            order_times = delivery_times(z, b, promise_min, seed=SEED)
            total_orders = total_orders + len(order_times)

            # count the orders that arrived after the promise
            for t in order_times:
                if t > promise_min:
                    late_orders = late_orders + 1

    # turn the late count into a percentage of all orders
    return late_orders / total_orders * 100

In [21]:
# Test the four cases with a 45-minute promise
print("Far West, Fri/Sat eve:", round(late_percentage("Far West", "Fri/Sat eve", 45), 1), "%")
print("All zones, Lunch:     ", round(late_percentage("all", "Lunch", 45), 1), "%")
print("North, all blocks:    ", round(late_percentage("North", "all", 45), 1), "%")
print("All zones, all blocks:", round(late_percentage("all", "all", 45), 1), "%")

Far West, Fri/Sat eve: 36.7 %
All zones, Lunch:      3.3 %
North, all blocks:     6.8 %
All zones, all blocks: 8.3 %


## Part 1b: Ranking by late rate (45-minute promise)

In [22]:
# build a table with one row per zone + time block combination
# each row looks like: [zone, time block, late %]
late_table = []
for z in ZONES:
    for b in TIME_BLOCKS:
        # late % for just this one zone and time block
        rate = late_percentage(z, b, PROMISE)
        late_table.append([z, b, rate])


# sort by the third item in each row (the late %), highest first
def get_value(row):
    return row[2]

late_table.sort(key=get_value, reverse=True)

# print the header row
# :<6, :<11, :<14 left-align each column and pad it to that width so the columns line up
print(f"{'Rank':<6}{'Zone':<11}{'Time block':<14}Late %")

# print each row with its rank (1 = worst late rate)
rank = 1
for row in late_table:
    # :.1f rounds the late % to 1 decimal place
    print(f"{rank:<6}{row[0]:<11}{row[1]:<14}{row[2]:.1f}%")
    rank = rank + 1

Rank  Zone       Time block    Late %
1     Far West   Fri/Sat eve   36.7%
2     North      Fri/Sat eve   16.7%
3     Far West   Weekday eve   11.2%
4     Central    Fri/Sat eve   6.8%
5     North      Weekday eve   6.0%
6     Far West   Other         4.7%
7     Central    Weekday eve   4.2%
8     North      Lunch         3.7%
9     Central    Lunch         3.5%
10    Central    Other         2.6%
11    North      Other         2.5%
12    Far West   Lunch         2.5%


**Result:** Far West on Fri/Sat evenings is by far the worst, with 36.7% of orders late. North Fri/Sat eve comes next at 16.7%. Every other pair is at 11.2% or below.

## Part 1c: Average delivery time

In [23]:
def average_delivery(zone, block, promise_min):
    # works out the average delivery time (in minutes) across the chosen orders
    # zone and block can each be one value or "all"
    total_minutes = 0
    total_orders = 0

    # go through every zone and time block combination we picked
    for z in pick_zones(zone):
        for b in pick_blocks(block):
            # simulate delivery times for this zone and block
            # (a fixed SEED gives the same results on every run)
            order_times = delivery_times(z, b, promise_min, seed=SEED)

            # add up all the delivery minutes and count the orders
            total_minutes = total_minutes + order_times.sum()
            total_orders = total_orders + len(order_times)

    # average = total minutes / number of orders
    return total_minutes / total_orders

In [24]:
# Test the four cases with a 45-minute promise
print("Far West, Fri/Sat eve:", round(average_delivery("Far West", "Fri/Sat eve", 45), 1), "min")
print("All zones, Lunch:     ", round(average_delivery("all", "Lunch", 45), 1), "min")
print("Far West, all blocks: ", round(average_delivery("Far West", "all", 45), 1), "min")
print("All zones, all blocks:", round(average_delivery("all", "all", 45), 1), "min")

Far West, Fri/Sat eve: 42.9 min
All zones, Lunch:      28.2 min
Far West, all blocks:  35.3 min
All zones, all blocks: 31.2 min


## Part 1d: Ranking by average delivery time (45-minute promise)

In [25]:
# build a table with one row per zone + time block combination
# each row looks like: [zone, time block, average delivery time]
avg_table = []
for z in ZONES:
    for b in TIME_BLOCKS:
        # average delivery time for just this one zone and time block
        avg = average_delivery(z, b, PROMISE)
        avg_table.append([z, b, avg])

# reuse get_value from earlier to sort by the third item (the average), slowest first
avg_table.sort(key=get_value, reverse=True)

# print the header row
# :<6, :<11, :<14 left-align each column and pad it to that width so the columns line up
print(f"{'Rank':<6}{'Zone':<11}{'Time block':<14}Avg time")

# print each row with its rank (1 = slowest average delivery)
rank = 1
for row in avg_table:
    # :.1f rounds the average to 1 decimal place
    print(f"{rank:<6}{row[0]:<11}{row[1]:<14}{row[2]:.1f} min")
    rank = rank + 1

Rank  Zone       Time block    Avg time
1     Far West   Fri/Sat eve   42.9 min
2     North      Fri/Sat eve   36.8 min
3     Far West   Weekday eve   35.0 min
4     Central    Fri/Sat eve   32.1 min
5     North      Weekday eve   31.4 min
6     Far West   Lunch         30.6 min
7     Far West   Other         30.0 min
8     Central    Weekday eve   29.0 min
9     North      Lunch         28.7 min
10    North      Other         27.7 min
11    Central    Lunch         26.8 min
12    Central    Other         26.0 min


## Part 1e: Which measure is more relevant to Rosa's decisions?

The **late rate** is more relevant to Rosa's decisions than the average delivery time.

**1. Rosa's costs come from late orders only.** Each late order leads to a refund and to lost future orders. An order delivered in 30 minutes costs Rosa the same as one delivered in 40 minutes: nothing. The late rate counts exactly the orders that cost money. The average does not.

**2. The average can sit below the promise while many orders are late.** Far West Fri/Sat eve averages 42.9 minutes, which is *under* the 45-minute promise, yet 36.7% of its orders are late. Overall, the average is only 31.2 minutes, but 8.3% of all orders still miss the promise. Looking at the average alone would make the service look fine.

**3. The average hides the slow tail.** Far West Lunch ranks 6th by average time (30.6 min) but last by late rate (2.5%). Its deliveries are moderately slow but rarely go past 45 minutes, so it costs Rosa very little. Using averages would overstate its problem.

**4. The rankings agree at the top, but the late rate shows how bad it is.** Far West Fri/Sat eve is worst on both measures. Only the late rate shows that more than a third of its orders are late, compared with 8.3% overall.

So Rosa should focus on late rates (and the number of late orders) when deciding where to change the promise.

# Part II: Exploring the best promise

## Part 2a: Total cost per late order

Each late order costs Rosa:
- a **refund** to the customer, and
- **churn**: the customer places fewer future orders, and each lost order loses the profit margin.

Total cost per late order = refund + (churn orders × margin per order)

In [26]:
def cost_of_one_late_order(cost_info):
    # works out how much one late order costs the business in total
    # cost_info is a dictionary holding the cost assumptions

    # money paid back to the customer for the late order
    refund = cost_info["refund"]

    # profit lost from future orders the customer won't place because of the late delivery
    # (number of lost orders x profit made on each order)
    lost_profit = cost_info["churn_orders"] * cost_info["margin"]

    # total cost = refund now + profit lost later
    total = refund + lost_profit
    return total

In [27]:
print("Refund per late order:      $", COSTS["refund"])
print("Lost profit from churn:     $", round(COSTS["churn_orders"] * COSTS["margin"], 2))
print("Total cost per late order:  $", round(cost_of_one_late_order(COSTS), 2))

Refund per late order:      $ 10.0
Lost profit from churn:     $ 16.2
Total cost per late order:  $ 26.2


## Part 2b: Choosing the best promise

**How to choose the range.** The promise involves a trade-off:
- If the promise is too short, almost every order is late, so refund and churn costs are larger than the profit.
- If the promise is too long, almost no orders are late, but fewer customers order, so profit shrinks.

The best promise is somewhere in between, so the range must be wide enough to contain the peak:
- The **low end** should be short enough that most orders are late and net profit is negative.
- The **high end** should be long enough that demand has clearly dropped and net profit is falling.

With the current promise, delivery times run from about 11 to 99 minutes (checked below), so I try **20 to 80 minutes in 5-minute steps**. 5-minute steps are also practical promises to advertise. After running the function, I check that the best promise is **not** at either end of the range. If it were, the real best could be outside the range, so I would widen it and run again.

**Method.** For each promise in the range:

net profit = (number of orders × margin) − (number of late orders × cost per late order)

`delivery_times` is called again for each promise, because a different promise changes demand, and demand changes delivery times. The function keeps track of the promise with the highest net profit seen so far.

In [28]:
# Shortest and longest delivery times with the current 45-minute promise

# starting values: shortest starts very high and longest starts at 0,
# so the first real delivery time will replace both of them
shortest = 1000
longest = 0

# check every zone and time block combination
for z in ZONES:
    for b in TIME_BLOCKS:
        # simulate delivery times for this zone and block
        # (a fixed SEED gives the same results on every run)
        order_times = delivery_times(z, b, PROMISE, seed=SEED)

        # keep this group's fastest order if it beats the shortest so far
        if order_times.min() < shortest:
            shortest = order_times.min()

        # keep this group's slowest order if it beats the longest so far
        if order_times.max() > longest:
            longest = order_times.max()

# show the overall fastest and slowest delivery across all orders
print("Shortest delivery:", shortest, "min")
print("Longest delivery: ", longest, "min")

Shortest delivery: 11.4 min
Longest delivery:  99.4 min


In [29]:
def find_best_promise(zone, block, promise_list, cost_info):
    # tries each promised delivery time in promise_list and finds the one
    # that gives the highest profit for this zone and time block

    # cost of a single late order (refund + lost future profit)
    late_cost = cost_of_one_late_order(cost_info)

    # one row per promise tried: [promise, orders, late orders, profit]
    results = []

    # no best yet, so start both as None
    best_promise = None
    best_profit = None

    for p in promise_list:
        # simulate orders using p as the promised time
        # (the number of orders can change with the promise)
        order_times = delivery_times(zone, block, p, seed=SEED)
        num_orders = len(order_times)

        # count the orders that arrived after the promise
        num_late = 0
        for t in order_times:
            if t > p:
                num_late = num_late + 1

        # profit = profit from all orders - cost of the late ones
        profit = num_orders * cost_info["margin"] - num_late * late_cost
        results.append([p, num_orders, num_late, profit])

        # remember this promise if it is the best one so far
        if best_profit is None or profit > best_profit:
            best_promise = p
            best_profit = profit

    # send back the best promise, its profit, and the full results table
    return best_promise, best_profit, results

### Testing `find_best_promise` on Far West, Fri/Sat eve

In [30]:
# promise times to test: 20, 25, 30, ... 80 minutes
# (range stops before 85, so 80 is the last value)
promise_list = list(range(20, 85, 5))

# find the most profitable promise for the Far West zone on Friday/Saturday evenings
best, profit, results = find_best_promise("Far West", "Fri/Sat eve", promise_list, COSTS)

# print the header row
# :<9, :<8, :<6 left-align each column and pad it to that width so the columns line up
print(f"{'Promise':<9}{'Orders':<8}{'Late':<6}Net profit")

# print one row per promise tried
# each row is [promise, orders, late orders, profit]
for row in results:
    # :.1f rounds the profit to 1 decimal place
    print(f"{row[0]:<9}{row[1]:<8}{row[2]:<6}${row[3]:.1f}")

# blank line before the summary
print()
print("Recommended promise:", best, "minutes")
print("Net profit: $", round(profit, 1))

# check the best promise is not at either end of the range
# if it is, a promise outside the range might be even better,
# so we can't be sure we found the true best
if best == promise_list[0] or best == promise_list[-1]:
    print("Best promise is at the edge of the range - widen the range and try again.")
else:
    print("Best promise is inside the range, so the range is wide enough.")

Promise  Orders  Late  Net profit
20       241     239   $-4092.8
25       239     233   $-3953.6
30       237     218   $-3578.6
35       232     179   $-2601.8
40       224     133   $-1468.6
45       210     77    $-127.4
50       188     33    $827.4
55       158     8     $1212.4
60       121     2     $1036.6
65       84      0     $756.0
70       54      0     $486.0
75       32      0     $288.0
80       18      0     $162.0

Recommended promise: 55 minutes
Net profit: $ 1212.4
Best promise is inside the range, so the range is wide enough.


### Best promise for every zone and time block

In [31]:
# counts how many zone + time block pairs have their best promise
# at the very start or end of promise_list
edge_count = 0

# print the header row
# :<11, :<14 left-align each column and pad it to that width so the columns line up
print(f"{'Zone':<11}{'Time block':<14}{'Best promise':<14}Net profit")

# find the best promise for every zone and time block combination
for z in ZONES:
    for b in TIME_BLOCKS:
        # results isn't used here, we only need the best promise and its profit
        best, profit, results = find_best_promise(z, b, promise_list, COSTS)

        # :.1f rounds the profit to 1 decimal place
        print(f"{z:<11}{b:<14}{best:<14}${profit:.1f}")

        # if the best promise is at either end of the range,
        # a value outside the range might be even better
        if best == promise_list[0] or best == promise_list[-1]:
            edge_count = edge_count + 1

# blank line before the summary
print()

# 0 means the range was wide enough for every pair
print("Pairs where the best promise is at the edge of the range:", edge_count)

Zone       Time block    Best promise  Net profit
Central    Lunch         45            $2104.2
Central    Weekday eve   45            $2603.2
Central    Fri/Sat eve   50            $1399.0
Central    Other         45            $1579.0
North      Lunch         45            $1526.6
North      Weekday eve   45            $1857.0
North      Fri/Sat eve   50            $996.4
North      Other         45            $1335.2
Far West   Lunch         45            $1001.4
Far West   Weekday eve   50            $1210.8
Far West   Fri/Sat eve   55            $1212.4
Far West   Other         45            $1492.2

Pairs where the best promise is at the edge of the range: 0


### Recommendation for Rosa
- Keep the **45-minute** promise for most zone and time-block pairs.
- Extend to **50 minutes** for Central Fri/Sat eve, North Fri/Sat eve and Far West Weekday eve.
- Extend to **55 minutes** for Far West Fri/Sat eve. At 45 minutes this slot loses money (about −$127). At 55 minutes it earns about $1,212.
- None of the best promises is at the edge of the 20–80 minute range, so the range was wide enough.

##Part III: Creating a Web App

 **Github Respistory**: https://github.com/jaineelshah10/Rosa-Pizza-Delivery.git
-**Deployed App**:

The app reuses the `best_promise` function from Part II. The user selects a zone and time block, sets the range of promises to try, adjusts margin, churn and refund, and clicks a button to see the recommended promise, a net-profit chart and the full results table.


##AI Appendix

###How I used AI
- Understanding the Assignment : I used claude to help me understand the assignment better by breaking it down into smaller steps and an explanation for each step.
- Writing the code: I used claude to help me understand the logic behind the code. Claude gave me the step by step format on how to approach the code. It gave me some sample code, I used that to analyze that and use the same logic to write the code for this assignment.
- Debugging: For certain parts, there was some errors that came up, such as; TypeError: type numpy.ndarray doesn't define_round_method. Claude help me explain that I was calling delivery_time instead of my own avg_time fucntion. So I fixed the test calls.

###Key Prompts
1. Tell me step by step how we would do this assignment, (with the assignment PDF attached
2. Tell how to approach the code to complete the assingment. How can I use that logic to code this assignment.
3. "I'm done Part II, tell me what I need to do for Part III."
4. For this "a. (4 points) Each late order is costly for Rosa.
- Direct cost due to refund issued to customers.
- Each late order leads to customer churn: the customer places fewer orders in the future, and
each lost order is lost profit.
You may find the refund cost and number of churns per late order, and the profit margin per
order in the dictionary COSTS .
Write a function to calculate the total cost per late order." How would I approach this, tell me a step by step breakdown to do this?
6. Explain how would I be able to create the github respistory and deploying the app.



